# BODHI Symptom Checker — v8 (Hybrid)

## Configuration

All tunable parameters. Change behavior here.


In [47]:
LIKELIHOOD_SCORES = {
    'very_high': 0.8,
    'high': 0.65,
    'medium': 0.5,
    'low': 0.35,
    'rare': 0.2,
    'zero': 0.0
}

LIKELIHOOD_SCORES_COMPRESSED = {
    'very_high': 0.8,
    'high': 0.65,
    'medium': 0.5,
    'low': 0.35,
    'rare': 0.2,
    'zero': 0.0
}

TRIAGE_GUIDANCE = {
      'emergency': {
          'urgency': 'immediate',
          'action': 'Based on your symptoms, we strongly advise you to visit the nearest Emergency Room (ER) as soon as possible.',
          'followup': 'Do not wait for a scheduled appointment. Seek immediate medical attention.',
          'tone': 'serious',
      },
      'worrisome': {
          'urgency': 'soon',
          'action': 'Based on your symptoms, we recommend that you book an appointment with a doctor within the next 1-2 days.',
          'followup': 'If your symptoms worsen before your appointment, consider visiting an urgent care center or ER.',
          'tone': 'cautious',
      },
      'opd_managed': {
          'urgency': 'routine',
          'action': 'Based on your symptoms, this condition can typically be managed through outpatient care.',
          'followup': 'We recommend visiting your primary care physician at your earliest convenience for proper evaluation.',
          'tone': 'reassuring',
      },
  }

# --- Condition-specific OPD follow-up guidance ---
# Overrides generic TRIAGE_GUIDANCE['opd_managed'] for specific conditions.
# LLM checks: if condition in CONDITION_OPD_GUIDANCE -> use this, else -> generic.
CONDITION_OPD_GUIDANCE = {
    82272006: {
        'name': 'Common Cold / Acute Rhinitis',
        'self_care': 'This condition can typically be managed at home with rest, fluids, and over-the-counter remedies.',
        'followup': 'If your symptoms do not improve within 10 days, or if they worsen at any point, please book an appointment with a doctor.',
        'reassess_after': '10 days',
    },
    235595009: {
        'name': 'Gastroesophageal reflux disease',
        'self_care': 'This condition can typically be managed with prescribed medication (such as PPIs) and lifestyle adjustments.',
        'followup': 'If you do not see improvement after 4 weeks of treatment, please book an appointment with your doctor for further evaluation.',
        'reassess_after': '4 weeks',
    },
    398057008: {
        'name': 'Tension-type headache',
        'self_care': 'This condition can typically be managed with over-the-counter pain relief and stress management techniques.',
        'followup': 'If your headaches persist or do not respond to medication, please consult a doctor for further assessment.',
        'reassess_after': 'no improvement with medication',
    },
}

# --- Elimination config ---
ELIM_CONFIG = {
    # YES-side: eliminate conditions not connected to ANY confirmed symptom
    'yes_eliminate_unconnected': True,
    
    # NO-side: eliminate conditions where P(S|C) is in this list
    # (they expected the symptom but patient doesn't have it)
    'no_eliminate_psc_levels': ['very_high', 'high'],
    
    # --- Protection layer (P(C|S)-based) ---
    # Conditions with strong P(C|S) get a counter instead of immediate elimination
    'protection_enabled': True,
    'protection_pcs_levels': ['very_high', 'high'],  # P(C|S) levels that trigger protection
    'protection_increment': 0.2,                      # added to counter per elimination attempt
    'protection_threshold': 0.6,                      # eliminated when counter >= this
    
    # --- Triage protection (plug-and-play, OFF by default) ---
    # Emergency conditions get automatic protection regardless of P(C|S)
    'triage_protection': False,                        # flip to True to enable
    'triage_always_protect': ['emergency'],            # triage levels that get protection
    'triage_protection_increment': 0.2,                # same counter system
}

# --- Ranking config ---
RANKING_CONFIG = {
    'demographic_method': 'addition',
    # Hybrid scoring (Option D):
    #   YES = +1/N  (coverage-based, N = condition's total connected roots)
    #   NO  = -1/N  (coverage-based)
    #   final = (coverage_yn + PCS_sum) * P(C) * G * A + RF_bonus
}


# =====================================================================
# QUESTION BUDGET (plug-and-play master control)
# =====================================================================
# Modes:
#   'full'         -> adaptive ON + red flag screening ON (within budget)
#   'no_adaptive'  -> adaptive OFF + red flag screening ON (within budget)
#   'base_only'    -> adaptive OFF + red flag screening OFF
#
# global_max applies across ALL phases (base + adaptive + red flag screening).
# adaptive_max and screening_max are per-phase sub-caps within that global ceiling.
QUESTION_BUDGET = {
    'mode': 'full',            # 'full' | 'no_adaptive' | 'base_only'
    'global_max': 13,          # hard ceiling across ALL question phases
    'adaptive_max': 2,         # max adaptive extra Qs (used in 'full' mode)
    'screening_max': 2,        # max red-flag screening Qs total (used in 'full' and 'no_adaptive')
}

# --- Questioning config ---
QUESTIONING_CONFIG = {
    'max_questions': 9,
    'top_n_discovered': 20,
    'min_pool_size': 3,          # stop if pool reaches this size
    'min_question_score': 0.0,   # skip questions below this score
    'score_threshold': 10,     # stop if any condition reaches this score
    'question_scoring_method': 'normalized_symptom_score',  # 'ev' or 'normalized_symptom_score'
    
    # --- Variant follow-up (plug-and-play) ---
    'variant_followup_enabled': True,   # flip to False to disable
    'max_variant_followups': 3,         # max variant Qs per confirmed symptom
    
    # --- Prerequisite questions (plug-and-play) ---
    # 'pre_screen': ask top N prerequisites before main loop
    # 'integrated': add prerequisites to main loop's scoring pool
    # 'both': pre-screen top N, then remaining compete in main loop
    # 'off': disabled (v4 behavior)
    'prerequisite_mode': 'pre_screen',
    'max_prerequisite_prescreens': 3,
    'prerequisite_elim_strengths': ['very_high', 'high'],
    
    # Delay prerequisites: 0 = ask immediately after variants (current behavior)
    # N > 0 = ask after N discovered symptom questions
    'prerequisite_after_n_discovered': 0,
    
    # CF count mode for prerequisite scoring:
    # 'static'  = count total symptom edges per condition from the ontology (v6 default)
    # 'dynamic' = count only symptoms confirmed by the user so far
    'prerequisite_cf_mode': 'static',

    # --- Adaptive extra questions (plug-and-play) ---
    # After the main loop ends, if the pool is still large OR top conditions
    # are too close in score, ask up to extra_question_count more questions.
    # Both triggers share one budget -- they do NOT stack.
    'adaptive_extra_enabled': True,
    'extra_question_count': 3,
    'extra_questions_pool_threshold': 30,
    'extra_questions_proximity_enabled': True,
    'proximity_top_n': 3,
    'proximity_threshold': 0.85,
}

# --- Relation type → human-readable question ---
RELATION_QUESTIONS = {
    'duration_since':    'How long have you had this symptom?',
    'location':          'Where is it located?',
    'severity':          'How severe is it?',
    'onset':             'How did it start?',
    'pain_type':         'What type of pain is it?',
    'characteristic':    'What are the characteristics?',
    'aggravated':        'What makes it worse?',
    'relieved':          'What makes it better?',
    'radiating':         'Does it radiate anywhere?',
    'temporal_pattern':  'What is the pattern over time?',
    'duration_lasts':    'How long does each episode last?',
    'laterality':        'Which side is it on?'
}

# --- Prerequisite → human-readable question ---
PREREQUISITE_QUESTIONS = {
    'Pregnant':                            'Are you currently pregnant?',
    'Alcoholic (current consumer)':        'Do you consume alcohol regularly?',
    'Diabetes Type 2':                     'Have you been diagnosed with Type 2 Diabetes?',
    'Diabetes mellitus type 1':            'Have you been diagnosed with Type 1 Diabetes?',
    'Hypertension':                        'Do you have high blood pressure (hypertension)?',
    'H/O Trauma':                          'Have you had any recent physical trauma or injury?',
    'Unprotected sexual intercourse':      'Have you had unprotected sexual intercourse recently?',
    'Uncircumcised':                       'Are you uncircumcised?',
    'H/O: chickenpox':                     'Have you had chickenpox before?',
    'H/O COVID-19':                        'Have you had COVID-19?',
    'Missed period':                       'Have you missed your period recently?',
    'recent childbirth':                   'Have you given birth recently?',
    'Exposure to cold':                    'Have you been exposed to extreme cold recently?',
    'High altitude':                       'Are you at or recently been at high altitude?',
    'Bite of domestic animal':             'Have you been bitten by an animal recently?',
    'Seizure':                             'Have you had a seizure recently?',
    'Taken a vaccine':                     'Have you recently received a vaccine?',
    'Intake of drug':                      'Have you recently started any new medication?',
    'Frequent use of nasal decongestants': 'Do you frequently use nasal decongestant sprays?',
    'H/O fracture':                        'Have you had a bone fracture before?',
    'Injury of head':                      'Have you had a recent head injury?',
    'Injury of shoulder region':           'Have you had a recent shoulder injury?',
    'Injury to elbow':                     'Have you had a recent elbow injury?',
    'Injury of hip':                       'Have you had a recent hip injury?',
    'Injury of ear':                       'Have you had a recent ear injury?',
}


# =====================================================================
# RED FLAG CONFIGURATION
# =====================================================================
RED_FLAG_CONFIG = {
    'enabled': True,
    'bonus': 1.0,
    'screening_top_n': 5,
}

RED_FLAG_MAP = {
    82272006: {
        'name': 'Common Cold / Acute Rhinitis',
        'triggers': [
            # Variant-only: must confirm the specific severity variant, not just root Fever
            {'flag': 'High fever (>39C)',            'match_roots': [], 'match_variants': ['Fever <severity> severe (104 < x)']},
            # Root match OK: ANY breathlessness during a cold is a red flag
            {'flag': 'Shortness of breath',          'match_roots': ['Breathlessness']},
            # Root match OK: ANY chest pain during a cold is a red flag
            {'flag': 'Chest pain',                   'match_roots': ['Chest pain']},
            # Variant-only: must confirm SEVERE ear pain or SEVERE headache specifically
            {'flag': 'Severe ear pain or headache',  'match_roots': [], 'match_variants': ['Headache <severity> severe', 'Ear pain <severity> severe']},
        ],
        'screening_questions': [],
        'referral_rules': None,
    },
    235595009: {
        'name': 'Gastroesophageal reflux disease',
        'triggers': [
            # Root match OK: any dysphagia is a red flag for GERD
            {'flag': 'Dysphagia',              'match_roots': ['Dysphagia']},
            # Root match OK: any chest pain with GERD is a red flag
            {'flag': 'Chest pain',             'match_roots': ['Chest pain']},
            # Fixed: KG root is 'Vomit' not 'Vomiting'
            {'flag': 'Vomiting',               'match_roots': ['Vomit']},
            # Variant-only: must confirm the specific bloody vomit variant
            {'flag': 'Blood in vomit',         'match_roots': [], 'match_variants': ['Vomit <char> bloody in vomit']},
            # Variant-only: must confirm severe specifically, not any abdominal pain
            {'flag': 'Abdominal pain (severe)', 'match_roots': [], 'match_variants': ['Abdominal pain <severity> severe']},
        ],
        'screening_questions': [
            {'flag': 'Odynophagia (pain on swallowing)',  'question': 'Do you experience pain when swallowing (odynophagia)?'},
            {'flag': 'Unexplained weight loss',           'question': 'Have you had unexplained weight loss recently?'},
            {'flag': 'Melena (black tarry stools)',       'question': 'Have you noticed black, tarry stools?'},
            {'flag': 'Anemia symptoms',                   'question': 'Have you been told you have anemia, or do you feel unusually tired and pale?'},
            {'flag': 'Pain radiating to arm or jaw',      'question': 'Does the pain radiate to your arm or jaw?'},
        ],
        'referral_rules': {
            'emergency': {
                'description': 'Severe chest pain with SOB/sweating/radiating, hematemesis, or melena',
                'any_flags': ['Blood in vomit', 'Melena (black tarry stools)'],
                'combo': {'flags': ['Chest pain', 'Pain radiating to arm or jaw'], 'require_all': True},
            },
            'urgent_clinic': {
                'description': 'Dysphagia, odynophagia, weight loss, or anemia',
                'any_flags': ['Dysphagia', 'Odynophagia (pain on swallowing)',
                              'Unexplained weight loss', 'Anemia symptoms'],
            },
            'routine': {
                'description': 'GERD confirmed but no alarm features',
            },
        },
    },
    398057008: {
        'name': 'Tension-type headache',
        'triggers': [
            # Variant-only: must confirm the specific thunderclap characteristic
            {'flag': 'Thunderclap headache',  'match_roots': [], 'match_variants': ['Headache <char> thunderclap']},
        ],
        'screening_questions': [
            # Neurological signs moved here — no KG variant exists, must ask directly
            {'flag': 'Headache with neurological signs',
             'question': 'Do you have any neurological symptoms such as vision changes, numbness, weakness, or difficulty speaking?'},
            {'flag': 'Triggered by cough/sneeze/straining/position change',
             'question': 'Is your headache triggered or worsened by coughing, sneezing, straining, or changing position?'},
            {'flag': 'New headache onset after age 50',
             'question': 'Is this a new type of headache that started after age 50?'},
        ],
        'referral_rules': None,
    },
}



# =====================================================================
# BUDGET-DRIVEN OVERRIDES (derived from QUESTION_BUDGET at load time)
# =====================================================================
_mode = QUESTION_BUDGET['mode']

if _mode == 'base_only':
    QUESTIONING_CONFIG['adaptive_extra_enabled'] = False
    RED_FLAG_CONFIG['enabled'] = False
elif _mode == 'no_adaptive':
    QUESTIONING_CONFIG['adaptive_extra_enabled'] = False
elif _mode == 'full':
    QUESTIONING_CONFIG['adaptive_extra_enabled'] = True

# Cap adaptive extra_question_count to budget
if QUESTIONING_CONFIG['adaptive_extra_enabled']:
    QUESTIONING_CONFIG['extra_question_count'] = min(
        QUESTIONING_CONFIG['extra_question_count'],
        QUESTION_BUDGET['adaptive_max']
    )


print("Configuration loaded.")
print(f"Elimination: protection={'ON' if ELIM_CONFIG['protection_enabled'] else 'OFF'}, "
      f"triage_protection={'ON' if ELIM_CONFIG['triage_protection'] else 'OFF'}")
print(f"Ranking: Hybrid (YES=+1/N, NO=-1/N)")
print(f"Stop: max_questions={QUESTIONING_CONFIG['max_questions']}, "
      f"min_pool={QUESTIONING_CONFIG['min_pool_size']}, "
      f"score_threshold={QUESTIONING_CONFIG['score_threshold']}")
print(f"Variant follow-up: {'ON' if QUESTIONING_CONFIG['variant_followup_enabled'] else 'OFF'}, "
      f"max_per_symptom={QUESTIONING_CONFIG['max_variant_followups']}")
print(f"Prerequisites: mode={QUESTIONING_CONFIG['prerequisite_mode']}, "
      f"max_prescreen={QUESTIONING_CONFIG['max_prerequisite_prescreens']}")
if QUESTIONING_CONFIG.get('adaptive_extra_enabled', False):
    print(f"Adaptive extra: ON (pool>{QUESTIONING_CONFIG['extra_questions_pool_threshold']}"
          f" | proximity top-{QUESTIONING_CONFIG['proximity_top_n']}"
          f" threshold={QUESTIONING_CONFIG['proximity_threshold']}"
          f" | max {QUESTIONING_CONFIG['extra_question_count']} extra Qs)")
else:
    print("Adaptive extra: OFF")
if RED_FLAG_CONFIG.get('enabled', False):
    print(f"Red flags: ON (bonus={RED_FLAG_CONFIG['bonus']}, "
          f"screening top-{RED_FLAG_CONFIG['screening_top_n']}, "
          f"{len(RED_FLAG_MAP)} conditions mapped)")
else:
    print("Red flags: OFF")
print(f"Budget: mode={QUESTION_BUDGET['mode']}, "
      f"global_max={QUESTION_BUDGET['global_max']}, "
      f"adaptive_max={QUESTION_BUDGET['adaptive_max']}, "
      f"screening_max={QUESTION_BUDGET['screening_max']}")


Configuration loaded.
Elimination: protection=ON, triage_protection=OFF
Ranking: Hybrid (YES=+1/N, NO=-1/N)
Stop: max_questions=9, min_pool=3, score_threshold=10
Variant follow-up: ON, max_per_symptom=3
Prerequisites: mode=pre_screen, max_prescreen=3
Adaptive extra: ON (pool>30 | proximity top-3 threshold=0.85 | max 2 extra Qs)
Red flags: ON (bonus=1.0, screening top-5, 3 conditions mapped)
Budget: mode=full, global_max=13, adaptive_max=2, screening_max=2


## Load Data

In [49]:
import pandas as pd
import math

nodes_symptom    = pd.read_csv('../csv/nodes_symptom.csv')
edges_present_in = pd.read_csv('../csv/edges_present_in.csv')
nodes_condition  = pd.read_csv('../csv/nodes_condition.csv')
edges_has_prerequisite = pd.read_csv('../csv/edges_has_prerequisite.csv')

# Normalize casing in BOTH likelihood columns
edges_present_in['likelihood_condition_given_symptom'] = edges_present_in['likelihood_condition_given_symptom'].str.lower()
edges_present_in['likelihood_symptom_given_condition'] = edges_present_in['likelihood_symptom_given_condition'].str.lower()

# Normalize overall_likelihood in nodes_condition
nodes_condition['overall_likelihood'] = nodes_condition['overall_likelihood'].str.lower()

# Normalize prerequisite edge strengths
edges_has_prerequisite['relation_strength'] = edges_has_prerequisite['relation_strength'].str.lower()
edges_has_prerequisite['relation_polarity'] = edges_has_prerequisite['relation_polarity'].str.lower()

# Pre-compute: which UUIDs have edges
UUIDS_WITH_EDGES = set(edges_present_in['symptom_uuid'].unique())

# Pre-compute: total condition weight for P(C) normalization
# P(C) = raw_weight / TOTAL_CONDITION_WEIGHT, so all P(C) values sum to ~1
TOTAL_CONDITION_WEIGHT = nodes_condition['overall_likelihood'].map(LIKELIHOOD_SCORES).sum()

print(f"Loaded: {len(nodes_symptom)} symptom rows, {len(edges_present_in)} edges, {len(nodes_condition)} conditions")
print(f"Prerequisites: {len(edges_has_prerequisite)} edges, "
      f"{edges_has_prerequisite['condition_snomed_id_dst'].nunique()} unique prerequisites")
print(f"Symptom UUIDs with edges: {len(UUIDS_WITH_EDGES)} / {nodes_symptom['uuid'].nunique()} "
      f"({len(UUIDS_WITH_EDGES)/nodes_symptom['uuid'].nunique()*100:.0f}%)")
print(f"Total condition weight (normalizer): {TOTAL_CONDITION_WEIGHT:.1f}")

# Pre-compute: number of connected roots per condition (for dynamic NO penalty)
CONDITION_ROOT_COUNTS = {}
for cid in nodes_condition['snomed_id'].unique():
    cond_uuids = edges_present_in[edges_present_in['condition_snomed_id']==cid]['symptom_uuid']
    n_roots = nodes_symptom[nodes_symptom['uuid'].isin(cond_uuids)]['root_snomed_name'].nunique()
    CONDITION_ROOT_COUNTS[cid] = max(n_roots, 1)
print(f"Root counts per condition computed for {len(CONDITION_ROOT_COUNTS)} conditions")

Loaded: 4037 symptom rows, 10352 edges, 779 conditions
Prerequisites: 53 edges, 25 unique prerequisites
Symptom UUIDs with edges: 2559 / 4037 (63%)
Total condition weight (normalizer): 356.7
Root counts per condition computed for 779 conditions


## Helper Functions

In [50]:
def get_age_bracket_column(age):
    brackets = [
        (1, 'likelihood_age_0_1'), (5, 'likelihood_age_1_5'),
        (12, 'likelihood_age_6_12'), (18, 'likelihood_age_13_18'),
        (30, 'likelihood_age_19_30'), (45, 'likelihood_age_30_45'),
        (60, 'likelihood_age_45_60'),
    ]
    for upper, col in brackets:
        if age <= upper:
            return col
    return 'likelihood_age_60_plus'

def get_gender_column(gender):
    return 'likelihood_male' if gender.upper() in ('M', 'MALE') else 'likelihood_female'

def get_root_uuid(root_snomed_name):
    row = nodes_symptom[
        (nodes_symptom['root_snomed_name'] == root_snomed_name) &
        (nodes_symptom['name'] == nodes_symptom['root_snomed_name'])
    ]
    return row.iloc[0]['uuid'] if len(row) > 0 else None

def get_condition_name(cid):
    row = nodes_condition[nodes_condition['snomed_id'] == cid]
    return row.iloc[0]['name'] if len(row) > 0 else 'Unknown'

def get_condition_triage(cid):
    row = nodes_condition[nodes_condition['snomed_id'] == cid]
    return row.iloc[0]['triage_level'] if len(row) > 0 else 'unknown'

def build_variant_questions(root_snomed_name):
    symptom_group = nodes_symptom[nodes_symptom['root_snomed_name'] == root_snomed_name].copy()
    if len(symptom_group) == 0:
        return []
    variants = symptom_group[
        (symptom_group['relation1_type'].notna()) &
        (symptom_group['uuid'].isin(UUIDS_WITH_EDGES))
    ]
    questions = []
    for relation_type, group in variants.groupby('relation1_type'):
        question_text = RELATION_QUESTIONS.get(relation_type, f"About the {relation_type}?")
        selection_type = group.iloc[0]['grouping1_selection_type']
        options = []
        for _, row in group.iterrows():
            if pd.notna(row.get('child2_name')):
                label = f"{row['child1_name']} > {row['child2_name']}"
            elif pd.notna(row.get('child1_name')):
                label = row['child1_name']
            else:
                label = row['name']
            options.append({'label': label, 'uuid': row['uuid'], 'name': row['name']})
        questions.append({
            'type': 'variant', 'root_name': root_snomed_name,
            'relation_type': relation_type, 'question': question_text,
            'selection_type': selection_type, 'options': options,
            'score_uuid': group.iloc[0]['uuid'], 'all_uuids': group['uuid'].tolist()
        })
    return questions

# def build_prerequisite_questions(candidate_pool, gender=None, confirmed_uuids=None, cf_mode='static'):
#     relevant = edges_has_prerequisite[
#         edges_has_prerequisite['condition_snomed_id_src'].isin(candidate_pool)
#     ]
#     if len(relevant) == 0:
#         return []
    
#     questions = []
#     for prereq_id, group in relevant.groupby('condition_snomed_id_dst'):
#         affected_conditions = set(group['condition_snomed_id_src'].unique()) & candidate_pool
#         if not affected_conditions:
#             continue
        
#         prereq_cond = nodes_condition[nodes_condition['snomed_id'] == prereq_id]
#         if len(prereq_cond) == 0:
#             continue
#         prereq_name = prereq_cond.iloc[0]['name']
        
#         if gender:
#             gender_col = get_gender_column(gender)
#             gender_weight = prereq_cond.iloc[0][gender_col]
#             if pd.notna(gender_weight) and float(gender_weight) == 0.0:
#                 continue
        
#         question_text = PREREQUISITE_QUESTIONS.get(
#             prereq_name,
#             f"Do you have or have you had: {prereq_name}?"
#         )
        
#         questions.append({
#             'type': 'prerequisite',
#             'prereq_id': prereq_id,
#             'prereq_name': prereq_name,
#             'question': question_text,
#             'affected_condition_ids': affected_conditions,
#             'num_affected': len(affected_conditions),
#             'score_uuid': f"prereq_{prereq_id}",
#         })
    
#     questions.sort(key=lambda x: x['num_affected'], reverse=True)
#     return questions

def build_prerequisite_questions(candidate_pool, gender=None, confirmed_uuids=None, cf_mode='static'):
    """
    Build and rank prerequisite questions by diagnostic value.

    Score for each prerequisite = sum over its connected conditions in pool of:
        relation_strength_score * P(C) * (1 / (count + 1))

    cf_mode controls what 'count' means:
      'static'  -> total symptom edges per condition from the ontology
      'dynamic' -> only symptoms the user has confirmed so far for that condition
    """
    STRENGTH_SCORES = {'very_high': 1.0, 'high': 0.8, 'medium': 0.6, 'low': 0.4, 'rare': 0.2, 'zero': 0.0}

    relevant = edges_has_prerequisite[
        edges_has_prerequisite['condition_snomed_id_src'].isin(candidate_pool)
    ]
    if len(relevant) == 0:
        return []

    if cf_mode == 'dynamic' and confirmed_uuids:
        confirmed_set = set(confirmed_uuids)
        confirmed_edges = edges_present_in[
            (edges_present_in['symptom_uuid'].isin(confirmed_set)) &
            (edges_present_in['condition_snomed_id'].isin(candidate_pool))
        ]
        cf_counts = confirmed_edges.groupby('condition_snomed_id')['symptom_uuid'].nunique().to_dict()
    else:
        cf_counts = edges_present_in.groupby('condition_snomed_id')['symptom_uuid'].nunique().to_dict()

    questions = []
    for prereq_id, group in relevant.groupby('condition_snomed_id_dst'):
        affected_conditions = set(group['condition_snomed_id_src'].unique()) & candidate_pool
        if not affected_conditions:
            continue

        prereq_cond = nodes_condition[nodes_condition['snomed_id'] == prereq_id]
        if len(prereq_cond) == 0:
            continue
        prereq_name = prereq_cond.iloc[0]['name']

        if gender:
            gender_col = get_gender_column(gender)
            gender_weight = prereq_cond.iloc[0][gender_col]
            if pd.notna(gender_weight) and float(gender_weight) == 0.0:
                continue

        question_text = PREREQUISITE_QUESTIONS.get(
            prereq_name,
            f"Do you have or have you had: {prereq_name}?"
        )

        # Score: weighted by relation_strength * P(C) * inverse CF
        prereq_score = 0.0
        for _, row in group[group['condition_snomed_id_src'].isin(affected_conditions)].iterrows():
            cid = row['condition_snomed_id_src']
            strength = STRENGTH_SCORES.get(row['relation_strength'], 0)
            cond_row = nodes_condition[nodes_condition['snomed_id'] == cid]
            pc = LIKELIHOOD_SCORES.get(cond_row.iloc[0]['overall_likelihood'], 0) if len(cond_row) > 0 else 0
            cf = cf_counts.get(cid, 0)
            prereq_score += strength * pc * (1.0 / (cf + 1))

        questions.append({
            'type': 'prerequisite',
            'prereq_id': prereq_id,
            'prereq_name': prereq_name,
            'question': question_text,
            'affected_condition_ids': affected_conditions,
            'num_affected': len(affected_conditions),
            'prereq_score': prereq_score,
            'score_uuid': f"prereq_{prereq_id}",
        })

    questions.sort(key=lambda x: x['prereq_score'], reverse=True)
    return questions

print("Helper functions defined.")

Helper functions defined.


## Function 1: Symptom Expansion

Takes a root symptom name, walks the knowledge graph to discover related symptoms via shared conditions.


In [51]:
def symptom_expansion(symptom_name):
    # Case-insensitive matching: find the correct casing from the data
    match = nodes_symptom[nodes_symptom['root_snomed_name'].str.lower() == symptom_name.lower()]
    if len(match) == 0:
        print(f"ERROR: Symptom '{symptom_name}' not found")
        return None
    
    # Use the actual casing from the data
    canonical_name = match.iloc[0]['root_snomed_name']
    symptom_group = nodes_symptom[nodes_symptom['root_snomed_name'] == canonical_name].copy()
    
    root_snomed_id = symptom_group.iloc[0]['root_snomed_id']
    all_uuids = symptom_group['uuid'].tolist()
    all_edges = edges_present_in[edges_present_in['symptom_uuid'].isin(all_uuids)].copy()
    condition_ids = set(all_edges['condition_snomed_id'].unique())
    
    # Reverse-engineer: discover other root symptoms through shared conditions
    discovered_roots = {}
    for cond_id in condition_ids:
        cond_edges = edges_present_in[edges_present_in['condition_snomed_id'] == cond_id]
        cond_symptoms = nodes_symptom[nodes_symptom['uuid'].isin(cond_edges['symptom_uuid'])][
            ['root_snomed_id', 'root_snomed_name']
        ].drop_duplicates('root_snomed_id')
        
        for _, row in cond_symptoms.iterrows():
            rid = row['root_snomed_id']
            if rid == root_snomed_id or rid in discovered_roots:
                continue
            discovered_roots[rid] = {
                'root_snomed_id': rid,
                'root_snomed_name': row['root_snomed_name'],
                'linked_conditions': set()
            }
        
        for rid in discovered_roots:
            if cond_id in edges_present_in[
                edges_present_in['symptom_uuid'].isin(
                    nodes_symptom[nodes_symptom['root_snomed_id'] == rid]['uuid']
                )
            ]['condition_snomed_id'].values:
                discovered_roots[rid]['linked_conditions'].add(cond_id)
    
    # Rank by total likelihood sum
    for rid, info in discovered_roots.items():
        root_uuid = get_root_uuid(info['root_snomed_name'])
        if root_uuid is None:
            info['total_likelihood_sum'] = 0
            continue
        root_edges = edges_present_in[
            (edges_present_in['symptom_uuid'] == root_uuid) &
            (edges_present_in['condition_snomed_id'].isin(condition_ids))
        ]
        info['total_likelihood_sum'] = root_edges['likelihood_condition_given_symptom'].map(
            LIKELIHOOD_SCORES
        ).sum()
    
    disc_rows = [{
        'root_snomed_id': info['root_snomed_id'],
        'root_snomed_name': info['root_snomed_name'],
        'num_shared_conditions': len(info['linked_conditions']),
        'total_likelihood_sum': info['total_likelihood_sum']
    } for info in discovered_roots.values()]
    
    discovered_df = pd.DataFrame(disc_rows).sort_values(
        'total_likelihood_sum', ascending=False
    ).reset_index(drop=True) if disc_rows else pd.DataFrame()
    
    print(f"Symptom: {canonical_name} (root_id: {root_snomed_id})")
    print(f"Variants: {len(symptom_group)} | Conditions: {len(condition_ids)} | Discovered symptoms: {len(discovered_df)}")
    
    return {
        'starting_root': {'root_snomed_id': root_snomed_id, 'root_snomed_name': canonical_name},
        'starting_variants_df': symptom_group,
        'condition_ids': condition_ids,
        'discovered_roots_df': discovered_df,
        'all_edges_df': all_edges
    }

print("Function 1 (symptom_expansion) defined.")

Function 1 (symptom_expansion) defined.


## Elimination Engine

Two-sided elimination with protection counter system.

**YES answer:** Eliminate conditions not connected to ANY confirmed symptom.

**NO answer:** Eliminate conditions where P(S|C) is high/very_high (they expected the symptom). Protected if P(C|S) is high/very_high, or if triage protection is enabled for emergency conditions.


In [52]:
def compute_yes_eliminations(symptom_uuid, pool, confirmed_uuids, protection_counters):
    """
    YES answer: eliminate conditions not connected to ANY confirmed symptom.
    Having a symptom is never evidence against a connected condition, even weakly.
    """
    eliminated = set()
    
    if ELIM_CONFIG['yes_eliminate_unconnected']:
        all_confirmed = set(confirmed_uuids) | {symptom_uuid}
        all_confirmed_edges = edges_present_in[edges_present_in['symptom_uuid'].isin(all_confirmed)]
        connected = set(all_confirmed_edges['condition_snomed_id'].unique()) & pool
        eliminated = pool - connected
    
    return eliminated, protection_counters


def compute_no_eliminations(symptom_uuid, pool, protection_counters):
    """
    NO answer: eliminate conditions that expected this symptom (high/very_high P(S|C)).
    Protected if P(C|S) is high/very_high (strong diagnostic link).
    Protected if triage_protection is ON and condition is emergency.
    Protection uses a counter — eliminated only when counter reaches threshold.
    """
    eliminated = set()
    edges = edges_present_in[
        (edges_present_in['symptom_uuid'] == symptom_uuid) &
        (edges_present_in['condition_snomed_id'].isin(pool))
    ]
    
    for _, edge in edges.iterrows():
        cid = edge['condition_snomed_id']
        psc = edge['likelihood_symptom_given_condition']
        pcs = edge['likelihood_condition_given_symptom']
        
        if psc not in ELIM_CONFIG['no_eliminate_psc_levels']:
            continue
        
        # Check P(C|S) protection
        pcs_protected = (
            ELIM_CONFIG['protection_enabled'] and
            pcs in ELIM_CONFIG['protection_pcs_levels']
        )
        
        # Check triage protection
        triage_protected = False
        if ELIM_CONFIG['triage_protection']:
            triage = nodes_condition[nodes_condition['snomed_id'] == cid]['triage_level'].values
            if len(triage) > 0 and triage[0] in ELIM_CONFIG['triage_always_protect']:
                triage_protected = True
        
        if pcs_protected or triage_protected:
            increment = ELIM_CONFIG['protection_increment']
            if triage_protected and not pcs_protected:
                increment = ELIM_CONFIG['triage_protection_increment']
            counter = protection_counters.get(cid, 0) + increment
            protection_counters[cid] = counter
            if counter >= ELIM_CONFIG['protection_threshold']:
                eliminated.add(cid)
        else:
            eliminated.add(cid)
    
    return eliminated, protection_counters


def compute_question_value(symptom_uuid, pool, confirmed_uuids, protection_counters, elim_config, questioning_config):
    """
    Score a question using Expected Value.
    
    P(S) = sum of P(S|Ci) * P(Ci) for all conditions Ci connected to S in pool
    where P(Ci) = LIKELIHOOD_SCORES[overall_likelihood] / TOTAL_CONDITION_WEIGHT
    
    P(YES) = P(S),  P(NO) = 1 - P(S)
    Expected Elimination = P(YES) * yes_elim + P(NO) * no_elim
    Score = Expected Elimination / pool_size
    
    Returns: (score, yes_elim_count, no_elim_count, connected_count, p_yes)
    """    
    edges = edges_present_in[
        (edges_present_in['symptom_uuid'] == symptom_uuid) &
        (edges_present_in['condition_snomed_id'].isin(pool))
    ]
    connected = set(edges['condition_snomed_id'].unique())
    if not connected:
        return 0.0, 0, 0, 0, 0.0
    
    pool_size = len(pool)
    method = questioning_config.get('question_scoring_method', 'ev')
    
    # ---- Normalized Symptom Score ----
    if method == 'normalized_symptom_score':
    # FIXED score: compute against ALL conditions, not just pool
        all_edges = edges_present_in[
            edges_present_in['symptom_uuid'] == symptom_uuid
        ]
        symptom_score = 0.0
        for _, edge in all_edges.iterrows():
            cid = edge['condition_snomed_id']
            psc_val = LIKELIHOOD_SCORES_COMPRESSED.get(edge['likelihood_symptom_given_condition'], 0)
            cond = nodes_condition[nodes_condition['snomed_id'] == cid]
            if len(cond) > 0:
                pc_val = LIKELIHOOD_SCORES_COMPRESSED.get(cond.iloc[0]['overall_likelihood'], 0)
            else:
                pc_val = 0
            symptom_score += psc_val * pc_val

            '''
            If wanted to calculate the P(S) over only the connedted conditions in the pool, we would do this:
            '''
    # if method == 'normalized_symptom_score':
    #     symptom_score = 0.0
    #     for _, edge in edges.iterrows():
    #         cid = edge['condition_snomed_id']
    #         psc_val = LIKELIHOOD_SCORES_COMPRESSED.get(edge['likelihood_symptom_given_condition'], 0)
    #         cond = nodes_condition[nodes_condition['snomed_id'] == cid]
    #         if len(cond) > 0:
    #             pc_val = LIKELIHOOD_SCORES_COMPRESSED.get(cond.iloc[0]['overall_likelihood'], 0)
    #         else:
    #             pc_val = 0
    #         symptom_score += psc_val * pc_val
        
        # normalized_score = symptom_score / len(connected)
        
        # yes/no elim counts (for logging only, not used in scoring)
        all_conf = set(confirmed_uuids) | {symptom_uuid}
        yes_connected = set(
            edges_present_in[edges_present_in['symptom_uuid'].isin(all_conf)]['condition_snomed_id'].unique()
        ) & pool
        yes_elim = len(pool - yes_connected)
        
        no_elim_set = set()
        temp_prot = dict(protection_counters)
        for _, edge in edges.iterrows():
            cid = edge['condition_snomed_id']
            psc = edge['likelihood_symptom_given_condition']
            pcs = edge['likelihood_condition_given_symptom']
            if psc in elim_config['no_eliminate_psc_levels']:
                pcs_prot = elim_config['protection_enabled'] and pcs in elim_config['protection_pcs_levels']
                triage_prot = False
                if elim_config['triage_protection']:
                    t = nodes_condition[nodes_condition['snomed_id'] == cid]['triage_level'].values
                    if len(t) > 0 and t[0] in elim_config['triage_always_protect']:
                        triage_prot = True
                if pcs_prot or triage_prot:
                    c = temp_prot.get(cid, 0) + elim_config['protection_increment']
                    temp_prot[cid] = c
                    if c >= elim_config['protection_threshold']:
                        no_elim_set.add(cid)
                else:
                    no_elim_set.add(cid)
        no_elim = len(no_elim_set)
        
        p_yes = symptom_score  # raw sum before normalization, for logging
        
        return symptom_score, yes_elim, no_elim, len(connected), p_yes
    
    # ---- Expected Value (original) ----
    else:
        p_yes = 0.0
        for _, edge in edges.iterrows():
            cid = edge['condition_snomed_id']
            psc_val = LIKELIHOOD_SCORES.get(edge['likelihood_symptom_given_condition'], 0)
            cond = nodes_condition[nodes_condition['snomed_id'] == cid]
            if len(cond) > 0:
                pc_raw = LIKELIHOOD_SCORES.get(cond.iloc[0]['overall_likelihood'], 0)
                pc_val = pc_raw / TOTAL_CONDITION_WEIGHT
            else:
                pc_val = 0
            p_yes += psc_val * pc_val
        p_yes = min(p_yes, 1.0)
        p_no = 1.0 - p_yes
        
        all_conf = set(confirmed_uuids) | {symptom_uuid}
        yes_connected = set(
            edges_present_in[edges_present_in['symptom_uuid'].isin(all_conf)]['condition_snomed_id'].unique()
        ) & pool
        yes_elim = len(pool - yes_connected)
        
        no_elim_set = set()
        temp_prot = dict(protection_counters)
        for _, edge in edges.iterrows():
            cid = edge['condition_snomed_id']
            psc = edge['likelihood_symptom_given_condition']
            pcs = edge['likelihood_condition_given_symptom']
            if psc in elim_config['no_eliminate_psc_levels']:
                pcs_prot = elim_config['protection_enabled'] and pcs in elim_config['protection_pcs_levels']
                triage_prot = False
                if elim_config['triage_protection']:
                    t = nodes_condition[nodes_condition['snomed_id'] == cid]['triage_level'].values
                    if len(t) > 0 and t[0] in elim_config['triage_always_protect']:
                        triage_prot = True
                if pcs_prot or triage_prot:
                    c = temp_prot.get(cid, 0) + elim_config['protection_increment']
                    temp_prot[cid] = c
                    if c >= elim_config['protection_threshold']:
                        no_elim_set.add(cid)
                else:
                    no_elim_set.add(cid)
        no_elim = len(no_elim_set)
        
        expected_elim = p_yes * yes_elim + p_no * no_elim
        score = expected_elim / pool_size if pool_size > 0 else 0
        return score, yes_elim, no_elim, len(connected), p_yes


print("Elimination engine defined (Expected Value scoring).")

Elimination engine defined (Expected Value scoring).


## Function 2: Dual-Track Questioning

Picks the best question by elimination power. YES/NO answers update both tracks:
- **Elimination track:** conditions removed from pool
- **Ranking track:** +1/-0.2 points accumulated per condition

Shows connected conditions for each question.


In [53]:
def ask_questions(expansion_result):
    """
    Function 2: Dual-track questioning with elimination + ranking.
    
    Returns: (confirmed_uuids, denied_uuids, surviving_pool,
              condition_points, protection_counters, question_log_df)
    """
    config_q = QUESTIONING_CONFIG
    config_r = RANKING_CONFIG

    # Budget-aware: base loop gets global_max minus reserved adaptive/screening slots
    _budget = QUESTION_BUDGET
    _effective_base_max = _budget['global_max']
    if _budget['mode'] == 'full':
        _effective_base_max -= _budget['adaptive_max']
    if _budget['mode'] in ('full', 'no_adaptive') and RED_FLAG_CONFIG.get('enabled', False):
        _effective_base_max -= _budget['screening_max']
    _effective_base_max = max(_effective_base_max, 1)
    # Use the smaller of config max_questions and budget-derived cap
    _base_max = min(config_q['max_questions'], _effective_base_max)

    def _coverage_penalty(cid):
        """NO penalty: -(1/N) where N = condition's total connected roots."""
        return -(1.0 / CONDITION_ROOT_COUNTS.get(cid, 1))
    prereq_mode = config_q.get('prerequisite_mode', 'off')
    
    starting_variants = expansion_result['starting_variants_df']
    root_name = expansion_result['starting_root']['root_snomed_name']
    root_snomed_id = expansion_result['starting_root']['root_snomed_id']
    candidate_pool = set(expansion_result['condition_ids'])
    
    confirmed_uuids = []
    denied_uuids = []
    protection_counters = {}
    condition_points = {cid: 0.0 for cid in candidate_pool}
    question_log = []
    
    print("=" * 70)
    print(f"  DUAL-TRACK QUESTIONING FOR: {root_name}")
    print(f"  Starting pool: {len(candidate_pool)} conditions")
    if config_q.get('variant_followup_enabled', False):
        print(f"  Variant follow-up: ON (max {config_q.get('max_variant_followups', 3)} per symptom)")
    if prereq_mode != 'off':
        print(f"  Prerequisites: mode={prereq_mode}")
    print("=" * 70)
    
    # Auto-confirm root symptom
    root_row = starting_variants[starting_variants['name'] == starting_variants['root_snomed_name']]
    if len(root_row) > 0:
        root_uuid = root_row.iloc[0]['uuid']
        confirmed_uuids.append(root_uuid)
        root_edges = edges_present_in[
            (edges_present_in['symptom_uuid'] == root_uuid) &
            (edges_present_in['condition_snomed_id'].isin(candidate_pool))
        ]
        for _, edge in root_edges.iterrows():
            cid_r = edge['condition_snomed_id']
            condition_points[cid_r] += 1.0 / CONDITION_ROOT_COUNTS.get(cid_r, 1)
        print(f"  Root confirmed: {root_name}")
    
    # ---- Helper: compute full final score for a condition ----
    def _full_score(cid):
        yn = condition_points.get(cid, 0)
        ce = edges_present_in[
            (edges_present_in['symptom_uuid'].isin(confirmed_uuids)) &
            (edges_present_in['condition_snomed_id'] == cid)
        ]
        pcs = ce['likelihood_condition_given_symptom'].map(LIKELIHOOD_SCORES).sum() if len(ce) > 0 else 0
        cond = nodes_condition[nodes_condition['snomed_id'] == cid]
        a_w = cond[get_age_bracket_column(age_input_global)].values[0] if len(cond) > 0 else 1.0
        g_w = cond[get_gender_column(gender_input_global)].values[0] if len(cond) > 0 else 1.0
        a_w = a_w if pd.notna(a_w) else 1.0
        g_w = g_w if pd.notna(g_w) else 1.0
        return yn + pcs + a_w + g_w
    
    # ---- Helper: ask a single variant question ----
    def _ask_one_variant(vq, is_followup=False):
        tag = "follow-up variant" if is_followup else "variant"
        
        vq_score, vq_ye, vq_ne, vq_conn, vq_pyes = compute_question_value(
            vq['score_uuid'], candidate_pool, confirmed_uuids,
            protection_counters, ELIM_CONFIG, QUESTIONING_CONFIG
        )
        
        vq_edges = edges_present_in[
            (edges_present_in['symptom_uuid'] == vq['score_uuid']) &
            (edges_present_in['condition_snomed_id'].isin(candidate_pool))
        ]
        vq_connected_cids = set(vq_edges['condition_snomed_id'].unique())
        
        vq_connected_display = []
        for cid in list(vq_connected_cids)[:5]:
            cname = get_condition_name(cid)[:30]
            cscore = _full_score(cid)
            vq_connected_display.append(f"{cname} ({cscore:.1f})")
        vq_connected_str = ', '.join(vq_connected_display)
        if len(vq_connected_cids) > 5:
            vq_connected_str += '...'
        
        options = vq['options']
        options_str = ", ".join(f"{i+1}.{o['label']}" for i, o in enumerate(options))
        mode = "pick ONE, 0=none" if vq['selection_type'] == 's' else "pick comma-separated, 0=none"
        
        print(f"\n  Q{questions_asked} [{tag}] (score:{vq_score:.3f} | P(YES):{vq_pyes:.3f} | YES_elim:{vq_ye} NO_elim:{vq_ne})")
        print(f"    {vq['question']}")
        for i, o in enumerate(options, 1):
            print(f"      {i}. {o['label']}")
        print(f"    Connected to: {vq_connected_str}")
        
        user_input = input(f"{vq['root_name']} - {vq['question']} ({options_str}) [{mode}]: ").strip()
        
        if user_input in ('0', ''):
            for o in options:
                denied_uuids.append(o['uuid'])
            no_edges = edges_present_in[
                (edges_present_in['symptom_uuid'] == vq['score_uuid']) &
                (edges_present_in['condition_snomed_id'].isin(vq_connected_cids))
            ]
            for _, edge in no_edges.iterrows():
                cid = edge['condition_snomed_id']
                if cid in condition_points:
                    condition_points[cid] += _coverage_penalty(cid)
            eliminated, _ = compute_no_eliminations(
                vq['score_uuid'], candidate_pool, protection_counters
            )
            candidate_pool.difference_update(eliminated)
            if eliminated:
                print(f"    -> Skipped. Eliminated {len(eliminated)}:")
                for cid in eliminated:
                    print(f"      X {get_condition_name(cid)} ({get_condition_triage(cid)})")
            else:
                print(f"    -> Skipped.")
        else:
            try:
                selected_uuids = set()
                if vq['selection_type'] == 's':
                    idx = int(user_input) - 1
                    if 0 <= idx < len(options):
                        confirmed_uuids.append(options[idx]['uuid'])
                        selected_uuids.add(options[idx]['uuid'])
                        print(f"    -> Confirmed: {options[idx]['label']}")
                else:
                    for idx in (int(x.strip())-1 for x in user_input.split(',')):
                        if 0 <= idx < len(options):
                            confirmed_uuids.append(options[idx]['uuid'])
                            selected_uuids.add(options[idx]['uuid'])
                            print(f"    -> Confirmed: {options[idx]['label']}")
                for o in options:
                    if o['uuid'] not in selected_uuids:
                        denied_uuids.append(o['uuid'])
                yes_edges = edges_present_in[
                    (edges_present_in['symptom_uuid'] == vq['score_uuid']) &
                    (edges_present_in['condition_snomed_id'].isin(vq_connected_cids))
                ]
                for _, edge in yes_edges.iterrows():
                    cid = edge['condition_snomed_id']
                    if cid in condition_points:
                        condition_points[cid] += 1.0 / CONDITION_ROOT_COUNTS.get(cid, 1)
                eliminated, _ = compute_yes_eliminations(
                    vq['score_uuid'], candidate_pool, confirmed_uuids, protection_counters
                )
                candidate_pool.difference_update(eliminated)
                if eliminated:
                    print(f"    Eliminated {len(eliminated)} unconnected conditions")
            except ValueError:
                print("    -> Invalid input, skipping")
        
        question_log.append({
            'order': questions_asked,
            'type': tag,
            'symptom': vq['root_name'],
            'relation_type': vq['relation_type'] or '-',
            'score': round(vq_score, 3),
            'p_yes': round(vq_pyes, 3),
            'yes_elim': vq_ye,
            'no_elim': vq_ne,
            'connected_conditions': ', '.join(get_condition_name(c) for c in vq_connected_cids),
            'pool_after': len(candidate_pool)
        })
        
        print(f"    Pool: {len(candidate_pool)} conditions remaining")
    
    # ---- Helper: ask a single prerequisite question ----
    def _ask_one_prerequisite(pq):
        affected_in_pool = pq['affected_condition_ids'] & candidate_pool
        if not affected_in_pool:
            return False
        
        affected_display = []
        for cid in list(affected_in_pool)[:5]:
            cname = get_condition_name(cid)[:30]
            cscore = _full_score(cid)
            affected_display.append(f"{cname} ({cscore:.1f})")
        affected_str = ', '.join(affected_display)
        if len(affected_in_pool) > 5:
            affected_str += '...'
        
        print(f"\n  Q{questions_asked} [prerequisite] (affects {len(affected_in_pool)} pool conditions)")
        print(f"    {pq['question']}")
        print(f"    Relevant to: {affected_str}")
        
        answer = input(f"{pq['question']} (y/n): ").strip().lower()
        
        to_eliminate = set()
        if answer in ('y', 'yes'):
            for cid in affected_in_pool:
                if cid in condition_points:
                    condition_points[cid] += 1.0 / CONDITION_ROOT_COUNTS.get(cid, 1)
            print(f"    -> Yes. {len(affected_in_pool)} conditions supported.")
        else:
            elim_strengths = config_q.get('prerequisite_elim_strengths', ['very_high', 'high'])
            for _, row in edges_has_prerequisite[
                (edges_has_prerequisite['condition_snomed_id_dst'] == pq['prereq_id']) &
                (edges_has_prerequisite['condition_snomed_id_src'].isin(candidate_pool))
            ].iterrows():
                if row['relation_strength'] in elim_strengths:
                    to_eliminate.add(row['condition_snomed_id_src'])
            candidate_pool.difference_update(to_eliminate)
            for cid in affected_in_pool:
                if cid in condition_points:
                    condition_points[cid] += _coverage_penalty(cid)
            if to_eliminate:
                print(f"    -> No. Eliminated {len(to_eliminate)} conditions:")
                for cid in to_eliminate:
                    print(f"      X {get_condition_name(cid)} ({get_condition_triage(cid)})")
            else:
                print(f"    -> No.")
        
        question_log.append({
            'order': questions_asked,
            'type': 'prerequisite',
            'symptom': pq['prereq_name'],
            'relation_type': 'prerequisite',
            'score': len(affected_in_pool),
            'p_yes': 0,
            'yes_elim': 0,
            'no_elim': len(to_eliminate),
            'connected_conditions': ', '.join(get_condition_name(c) for c in affected_in_pool),
            'pool_after': len(candidate_pool)
        })
        
        print(f"    Pool: {len(candidate_pool)} conditions remaining")
        return True
    
    # ---- Build askable question pool ----
    askable = []
    
    # Part A: Variant questions (starting symptom)
    variants = starting_variants[
        (starting_variants['relation1_type'].notna()) &
        (starting_variants['uuid'].isin(UUIDS_WITH_EDGES))
    ]
    for relation_type, group in variants.groupby('relation1_type'):
        question_text = RELATION_QUESTIONS.get(relation_type, f"About the {relation_type}?")
        selection_type = group.iloc[0]['grouping1_selection_type']
        options = []
        for _, row in group.iterrows():
            if pd.notna(row.get('child2_name')):
                label = f"{row['child1_name']} > {row['child2_name']}"
            elif pd.notna(row.get('child1_name')):
                label = row['child1_name']
            else:
                label = row['name']
            options.append({'label': label, 'uuid': row['uuid'], 'name': row['name']})
        askable.append({
            'type': 'variant', 'root_name': root_name,
            'relation_type': relation_type, 'question': question_text,
            'selection_type': selection_type, 'options': options,
            'score_uuid': group.iloc[0]['uuid'], 'all_uuids': group['uuid'].tolist()
        })
    
    # Part B: Discovered root symptoms
    discovered = expansion_result['discovered_roots_df'].head(config_q['top_n_discovered'])
    for _, row in discovered.iterrows():
        disc_uuid = get_root_uuid(row['root_snomed_name'])
        if disc_uuid is None or disc_uuid not in UUIDS_WITH_EDGES:
            continue
        askable.append({
            'type': 'discovered', 'root_name': row['root_snomed_name'],
            'relation_type': None,
            'question': f"Do you have '{row['root_snomed_name']}'?",
            'selection_type': None, 'options': None,
            'score_uuid': disc_uuid, 'all_uuids': [disc_uuid]
        })
    
    # Part C: Prerequisite questions (built once, used by pre-screen and/or integrated)
    all_prereq_qs = []
    if prereq_mode != 'off':
        cf_mode = config_q.get('prerequisite_cf_mode', 'static')
        all_prereq_qs = build_prerequisite_questions(candidate_pool, gender_input_global, confirmed_uuids, cf_mode)
    if prereq_mode in ('integrated', 'both'):
        askable.extend(all_prereq_qs)
    
    # ---- Questioning loop ----
    questions_asked = 0
    asked_uuids = set()
    
    # ---- Phase 1: Variant follow-up for STARTING symptom ----
    if config_q.get('variant_followup_enabled', False):
        starting_variant_qs = build_variant_questions(root_name)
        scored_starting = []
        for svq in starting_variant_qs:
            if svq['score_uuid'] in asked_uuids:
                continue
            sv_score = compute_question_value(
                svq['score_uuid'], candidate_pool, confirmed_uuids,
                protection_counters, ELIM_CONFIG, QUESTIONING_CONFIG
            )[0]
            scored_starting.append((sv_score, svq))
        scored_starting.sort(key=lambda x: x[0], reverse=True)
        
        max_fups = config_q.get('max_variant_followups', 3)
        fup_count = 0
        for _, svq in scored_starting:
            if fup_count >= max_fups:
                break
            if questions_asked >= _base_max:
                break
            if len(candidate_pool) <= config_q['min_pool_size']:
                break
            asked_uuids.add(svq['score_uuid'])
            questions_asked += 1
            _ask_one_variant(svq, is_followup=True)
            fup_count += 1
    
    # ---- Phase 2: Pre-screen prerequisites (if not delayed) ----
    prereq_delay = config_q.get('prerequisite_after_n_discovered', 0)
    prereqs_done = False
    
    if prereq_delay == 0 and prereq_mode in ('pre_screen', 'both') and all_prereq_qs:
        max_prescreens = config_q.get('max_prerequisite_prescreens', 3)
        prescreen_count = 0
        for pq in all_prereq_qs:
            if prescreen_count >= max_prescreens:
                break
            if questions_asked >= _base_max:
                break
            if len(candidate_pool) <= config_q['min_pool_size']:
                break
            affected_now = pq['affected_condition_ids'] & candidate_pool
            if not affected_now:
                continue
            asked_uuids.add(pq['score_uuid'])
            questions_asked += 1
            asked = _ask_one_prerequisite(pq)
            if asked:
                prescreen_count += 1
        prereqs_done = True
    
    # ---- Phase 3: Main questioning loop ----
    discovered_q_count = 0
    
    # ---- Helper: run delayed prerequisites ----
    def _run_delayed_prereqs():
        nonlocal questions_asked, prereqs_done, all_prereq_qs
        if prereqs_done or prereq_mode not in ('pre_screen', 'both'):
            return
        cf_mode = config_q.get('prerequisite_cf_mode', 'static')
        all_prereq_qs = build_prerequisite_questions(candidate_pool, gender_input_global, confirmed_uuids, cf_mode)
        if not all_prereq_qs:
            prereqs_done = True
            return
        prereqs_done = True
        max_prescreens = config_q.get('max_prerequisite_prescreens', 3)
        prescreen_count = 0
        for pq in all_prereq_qs:
            if prescreen_count >= max_prescreens:
                break
            if questions_asked >= _base_max:
                break
            if len(candidate_pool) <= config_q['min_pool_size']:
                break
            if pq['score_uuid'] in asked_uuids:
                continue
            affected_now = pq['affected_condition_ids'] & candidate_pool
            if not affected_now:
                continue
            asked_uuids.add(pq['score_uuid'])
            questions_asked += 1
            asked = _ask_one_prerequisite(pq)
            if asked:
                prescreen_count += 1
    
    while askable and questions_asked < _base_max:
        
        if len(candidate_pool) <= config_q['min_pool_size']:
            print(f"\n  -> Early stop: pool down to {len(candidate_pool)}")
            break
        
        if config_q['score_threshold'] is not None:
            max_score = max(condition_points[c] for c in candidate_pool) if candidate_pool else 0
            if max_score >= config_q['score_threshold']:
                print(f"\n  -> Early stop: condition reached Y/N score {max_score:.1f}")
                break
        
        scored = []
        for q in askable:
            if q['score_uuid'] in asked_uuids:
                continue
            if q['type'] == 'prerequisite':
                affected_now = q['affected_condition_ids'] & candidate_pool
                if affected_now:
                    scored.append((len(affected_now), 0, len(affected_now), len(affected_now), 0, q))
                continue
            score, ye, ne, conn, p_yes = compute_question_value(
                q['score_uuid'], candidate_pool, confirmed_uuids, protection_counters, ELIM_CONFIG, QUESTIONING_CONFIG
            )
            if score > config_q['min_question_score'] or ye > 0 or ne > 0:
                scored.append((score, ye, ne, conn, p_yes, q))
        
        if not scored:
            print(f"\n  -> No more valuable questions")
            break
        
        scored.sort(key=lambda x: x[0], reverse=True)
        score, ye, ne, conn, p_yes, q = scored[0]
        asked_uuids.add(q['score_uuid'])
        questions_asked += 1
        
        q_edges = edges_present_in[
            (edges_present_in['symptom_uuid'] == q['score_uuid']) &
            (edges_present_in['condition_snomed_id'].isin(candidate_pool))
        ]
        connected_cids = set(q_edges['condition_snomed_id'].unique())
        
        connected_display = []
        for cid in list(connected_cids)[:5]:
            cname = get_condition_name(cid)[:30]
            cscore = _full_score(cid)
            connected_display.append(f"{cname} ({cscore:.1f})")
        connected_str = ', '.join(connected_display)
        if len(connected_cids) > 5:
            connected_str += '...'
        
        # ---- Ask the question ----
        if q['type'] == 'variant':
            _ask_one_variant(q, is_followup=False)
        
        elif q['type'] == 'prerequisite':
            _ask_one_prerequisite(q)
        
        else:
            # Discovered symptom: yes/no
            print(f"\n  Q{questions_asked} [discovered] (score:{score:.3f} | P(YES):{p_yes:.3f} | YES_elim:{ye} NO_elim:{ne})")
            print(f"    Do you have '{q['root_name']}'?")
            print(f"    Connected to: {connected_str}")
            
            answer = input(f"Do you have '{q['root_name']}'? (y/n): ").strip().lower()
            
            if answer in ('y', 'yes'):
                confirmed_uuids.append(q['score_uuid'])
                print(f"    -> Confirmed: {q['root_name']}")
                for cid in connected_cids:
                    if cid in condition_points:
                        condition_points[cid] += 1.0 / CONDITION_ROOT_COUNTS.get(cid, 1)
                eliminated, protection_counters = compute_yes_eliminations(
                    q['score_uuid'], candidate_pool, confirmed_uuids, protection_counters
                )
            else:
                denied_uuids.append(q['score_uuid'])
                print(f"    -> No")
                for cid in connected_cids:
                    if cid in condition_points:
                        condition_points[cid] += _coverage_penalty(cid)
                eliminated, protection_counters = compute_no_eliminations(
                    q['score_uuid'], candidate_pool, protection_counters
                )
            
            candidate_pool -= eliminated
            if eliminated:
                print(f"    Eliminated {len(eliminated)}:")
                for cid in eliminated:
                    print(f"      X {get_condition_name(cid)} ({get_condition_triage(cid)})")
            
            question_log.append({
                'order': questions_asked,
                'type': q['type'],
                'symptom': q['root_name'],
                'relation_type': q['relation_type'] or '-',
                'score': round(score, 3),
                'p_yes': round(p_yes, 3),
                'yes_elim': ye,
                'no_elim': ne,
                'connected_conditions': ', '.join(get_condition_name(c) for c in connected_cids),
                'pool_after': len(candidate_pool)
            })
            
            print(f"    Pool: {len(candidate_pool)} conditions remaining")
            
            # Track discovered questions for delayed prerequisites
            discovered_q_count += 1
            if prereq_delay > 0 and discovered_q_count >= prereq_delay and not prereqs_done:
                _run_delayed_prereqs()
            
            # ---- Variant follow-up: when a discovered symptom is confirmed ----
            if (q['type'] == 'discovered' and
                answer in ('y', 'yes') and
                config_q.get('variant_followup_enabled', False)):
                
                followup_qs = build_variant_questions(q['root_name'])
                scored_fups = []
                for fq in followup_qs:
                    if fq['score_uuid'] in asked_uuids:
                        continue
                    fs = compute_question_value(
                        fq['score_uuid'], candidate_pool, confirmed_uuids,
                        protection_counters, ELIM_CONFIG, QUESTIONING_CONFIG
                    )[0]
                    scored_fups.append((fs, fq))
                scored_fups.sort(key=lambda x: x[0], reverse=True)
                
                max_fups = config_q.get('max_variant_followups', 3)
                fup_count = 0
                for _, fq in scored_fups:
                    if fup_count >= max_fups:
                        break
                    if questions_asked >= _base_max:
                        break
                    if len(candidate_pool) <= config_q['min_pool_size']:
                        break
                    asked_uuids.add(fq['score_uuid'])
                    questions_asked += 1
                    _ask_one_variant(fq, is_followup=True)
                    fup_count += 1
    
        # ---- Phase 4: Adaptive extra questions ----
    adaptive_triggered = False
    trigger_reasons = []
    extra_asked = 0

    if config_q.get('adaptive_extra_enabled', False):
        pool_threshold = config_q.get('extra_questions_pool_threshold', 30)
        pool_triggered = len(candidate_pool) > pool_threshold

        proximity_triggered = False
        if config_q.get('extra_questions_proximity_enabled', False) and len(candidate_pool) >= 2:
            top_n = config_q.get('proximity_top_n', 3)
            prox_thresh = config_q.get('proximity_threshold', 0.85)

            pool_scores = sorted([_full_score(cid) for cid in candidate_pool], reverse=True)
            leader = pool_scores[0]
            if leader > 0:
                cluster_size = 1
                for i in range(1, min(top_n, len(pool_scores))):
                    if pool_scores[i] / leader >= prox_thresh:
                        cluster_size += 1
                    else:
                        break
                if cluster_size >= 2:
                    proximity_triggered = True
                    trigger_reasons.append(
                        f"top-{cluster_size} within {prox_thresh:.0%} "
                        f"(scores: {', '.join(f'{s:.1f}' for s in pool_scores[:cluster_size])})"
                    )

        if pool_triggered:
            trigger_reasons.append(f"pool={len(candidate_pool)} > {pool_threshold}")

        if pool_triggered or proximity_triggered:
            adaptive_triggered = True
            extra_budget = min(config_q.get('extra_question_count', 3), _budget.get('adaptive_max', 3))

            print(f"\n  >> Adaptive extra questions triggered: {', '.join(trigger_reasons)}")
            print(f"     Asking up to {extra_budget} more questions...")

            while extra_asked < extra_budget and questions_asked < _budget['global_max']:
                if len(candidate_pool) <= config_q['min_pool_size']:
                    print(f"\n  -> Adaptive stop: pool down to {len(candidate_pool)}")
                    break

                scored = []
                for q in askable:
                    if q['score_uuid'] in asked_uuids:
                        continue
                    if q['type'] == 'prerequisite':
                        affected_now = q['affected_condition_ids'] & candidate_pool
                        if affected_now:
                            scored.append((len(affected_now), 0, len(affected_now), len(affected_now), 0, q))
                        continue
                    score, ye, ne, conn, p_yes = compute_question_value(
                        q['score_uuid'], candidate_pool, confirmed_uuids,
                        protection_counters, ELIM_CONFIG, QUESTIONING_CONFIG
                    )
                    if score > config_q['min_question_score'] or ye > 0 or ne > 0:
                        scored.append((score, ye, ne, conn, p_yes, q))

                if not scored:
                    print(f"\n  -> Adaptive stop: no more valuable questions")
                    break

                scored.sort(key=lambda x: x[0], reverse=True)
                score, ye, ne, conn, p_yes, q = scored[0]
                asked_uuids.add(q['score_uuid'])
                questions_asked += 1
                extra_asked += 1

                q_edges = edges_present_in[
                    (edges_present_in['symptom_uuid'] == q['score_uuid']) &
                    (edges_present_in['condition_snomed_id'].isin(candidate_pool))
                ]
                connected_cids = set(q_edges['condition_snomed_id'].unique())

                connected_display = []
                for cid in list(connected_cids)[:5]:
                    cname = get_condition_name(cid)[:30]
                    cscore = _full_score(cid)
                    connected_display.append(f"{cname} ({cscore:.1f})")
                connected_str = ', '.join(connected_display)
                if len(connected_cids) > 5:
                    connected_str += '...'

                if q['type'] == 'variant':
                    _ask_one_variant(q, is_followup=False)
                elif q['type'] == 'prerequisite':
                    _ask_one_prerequisite(q)
                else:
                    print(f"\n  Q{questions_asked} [adaptive-extra] (score:{score:.3f} | P(YES):{p_yes:.3f} | YES_elim:{ye} NO_elim:{ne})")
                    print(f"    Do you have '{q['root_name']}'?")
                    print(f"    Connected to: {connected_str}")

                    answer = input(f"Do you have '{q['root_name']}'? (y/n): ").strip().lower()

                    if answer in ('y', 'yes'):
                        confirmed_uuids.append(q['score_uuid'])
                        print(f"    -> Confirmed: {q['root_name']}")
                        for cid in connected_cids:
                            if cid in condition_points:
                                condition_points[cid] += 1.0 / CONDITION_ROOT_COUNTS.get(cid, 1)
                        eliminated, protection_counters = compute_yes_eliminations(
                            q['score_uuid'], candidate_pool, confirmed_uuids, protection_counters
                        )
                    else:
                        denied_uuids.append(q['score_uuid'])
                        print(f"    -> No")
                        for cid in connected_cids:
                            if cid in condition_points:
                                condition_points[cid] += _coverage_penalty(cid)
                        eliminated, protection_counters = compute_no_eliminations(
                            q['score_uuid'], candidate_pool, protection_counters
                        )

                    candidate_pool.difference_update(eliminated)
                    if eliminated:
                        print(f"    Eliminated {len(eliminated)} conditions")

                    question_log.append({
                        'q_number': questions_asked,
                        'type': 'adaptive-extra',
                        'root_name': q['root_name'],
                        'symptom_uuid': q['score_uuid'],
                        'answer': answer,
                        'score': round(score, 4),
                        'yes_elim': ye,
                        'no_elim': ne,
                        'connected_conditions': ', '.join(get_condition_name(c) for c in connected_cids),
                        'pool_after': len(candidate_pool)
                    })

                    print(f"    Pool: {len(candidate_pool)} conditions remaining")

            if extra_asked > 0:
                print(f"\n  >> Adaptive phase done: asked {extra_asked} extra question(s)")

    remaining = sum(1 for q in askable if q['score_uuid'] not in asked_uuids)

    print(f"\n{'=' * 70}")
    print(f"  Questions asked: {questions_asked} (budget max: {_budget['global_max']}) | Remaining possible: {remaining}")
    if adaptive_triggered:
        base = questions_asked - extra_asked
        print(f"    (base: {base} + adaptive: {extra_asked})")
    print(f"  Confirmed: {len(confirmed_uuids)} | Denied: {len(denied_uuids)}")
    print(f"  Surviving conditions: {len(candidate_pool)}")
    print(f"  Eliminated: {len(expansion_result['condition_ids']) - len(candidate_pool)}")
    print(f"{'=' * 70}")

    return (confirmed_uuids, denied_uuids, candidate_pool,
            condition_points, protection_counters, pd.DataFrame(question_log))


print("Function 2 (ask_questions) defined.")

Function 2 (ask_questions) defined.


## Function 3: Rank Conditions

Scores surviving conditions using the dual-track system:
- Y/N points from the questioning phase
- P(C|S) likelihood from confirmed symptoms
- Demographics (age + gender)


In [54]:
def rank_conditions(confirmed_uuids, denied_uuids, surviving_pool,
                    condition_points, age, gender):
    """
    Rank surviving conditions.
    
    Hybrid (Option D):
    final_score = (coverage_yn + sum(P(C|S) from confirmed)) * P(C) * age_weight * gender_weight
    coverage_yn = sum(+1/N for confirmed, -1/N for denied) accumulated during questioning
    
    Returns: (result_df, scoring_detail_df)
    """
    if not surviving_pool:
        print("No surviving conditions to rank.")
        return pd.DataFrame(), pd.DataFrame()
    
    # P(C|S) scores from confirmed symptoms
    confirmed_edges = edges_present_in[
        (edges_present_in['symptom_uuid'].isin(confirmed_uuids)) &
        (edges_present_in['condition_snomed_id'].isin(surviving_pool))
    ].copy()
    
    confirmed_edges['pcs_score'] = confirmed_edges[
        'likelihood_condition_given_symptom'
    ].map(LIKELIHOOD_SCORES).fillna(0)
    
    pcs_agg = confirmed_edges.groupby('condition_snomed_id').agg(
        pcs_total=('pcs_score', 'sum'),
        num_symptom_matches=('symptom_uuid', 'nunique')
    ).reset_index()
    
    # Demographics
    gender_col = get_gender_column(gender)
    age_col = get_age_bracket_column(age)
    
    # Build result for all surviving conditions
    rows = []
    for cid in surviving_pool:
        yn = condition_points.get(cid, 0)
        pcs_row = pcs_agg[pcs_agg['condition_snomed_id'] == cid]
        pcs = pcs_row['pcs_total'].values[0] if len(pcs_row) > 0 else 0
        matches = int(pcs_row['num_symptom_matches'].values[0]) if len(pcs_row) > 0 else 0
        
        cond = nodes_condition[nodes_condition['snomed_id'] == cid]
        name = cond['name'].values[0] if len(cond) > 0 else 'Unknown'
        triage = cond['triage_level'].values[0] if len(cond) > 0 else 'unknown'
        type_c = cond['type_condition'].values[0] if len(cond) > 0 else 'unknown'
        
        g_w = cond[gender_col].values[0] if len(cond) > 0 else 1.0
        a_w = cond[age_col].values[0] if len(cond) > 0 else 1.0
        g_w = g_w if pd.notna(g_w) else 1.0
        a_w = a_w if pd.notna(a_w) else 1.0
        
        # P(C): overall prevalence as a multiplier (0.2=rare .. 1.0=very_high)
        # Use the raw LIKELIHOOD_SCORES value directly — NOT divided by TOTAL_CONDITION_WEIGHT.
        # That normalizer is only for computing P(S) during question scoring, not for ranking.
        pc = LIKELIHOOD_SCORES.get(cond['overall_likelihood'].values[0], 0) if len(cond) > 0 else 0
        
        # Total connected roots for this condition
        total_roots = CONDITION_ROOT_COUNTS.get(cid, 1)

        if RANKING_CONFIG['demographic_method'] == 'multiplication':
            final = yn * pcs * g_w * a_w * pc if pcs > 0 else 0
        else:
            final = (yn + pcs) * pc * g_w * a_w

        
        rows.append({
            'condition_snomed_id': cid,
            'condition_name': name,
            'triage_level': triage,
            'type_condition': type_c,
            'num_symptom_matches': matches,
            'total_roots': total_roots,
            'match_coverage': round(matches / total_roots, 3) if total_roots > 0 else 0,
            'yn_points': round(yn, 2),
            'pcs_score': round(pcs, 2),
            'pc_weight': round(pc, 4),
            'gender_weight': round(g_w, 2),
            'age_weight': round(a_w, 2),
            'coverage_bonus': 0,
            'final_score': round(final, 4)
        })
    
    result = pd.DataFrame(rows).sort_values('final_score', ascending=False).reset_index(drop=True)
    
    # Scoring detail
    detail = confirmed_edges.merge(
        nodes_symptom[['uuid', 'name', 'root_snomed_name']],
        left_on='symptom_uuid', right_on='uuid', how='left'
    ).merge(
        nodes_condition[['snomed_id', 'name']],
        left_on='condition_snomed_id', right_on='snomed_id', how='left',
        suffixes=('_symptom', '_condition')
    )
    detail = detail[[
        'name_symptom', 'root_snomed_name', 'name_condition',
        'likelihood_condition_given_symptom', 'pcs_score'
    ]].copy()
    detail.columns = ['symptom_variant', 'root_symptom', 'condition_name',
                       'likelihood_text', 'pcs_score']
    detail = detail.sort_values(['condition_name', 'pcs_score'], ascending=[True, False]).reset_index(drop=True)
    
    print(f"Ranked {len(result)} surviving conditions")
    print(f"Demographics: gender={gender} ({gender_col}), age={age} ({age_col})")
    print(f"Method: {RANKING_CONFIG['demographic_method']}")
    
    return result, detail


print("Function 3 (rank_conditions) defined.")

Function 3 (rank_conditions) defined.


## Red Flag Post-Processing

In [55]:
def process_red_flags(confirmed_uuids, denied_uuids, surviving_pool,
                      condition_points, result_df):
    """
    Post-processing: scan confirmed symptoms for clinical red flags,
    ask screening questions for top-N conditions, apply bonus, determine referral.

    Called AFTER rank_conditions(). Modifies result_df in-place and returns
    a red_flag_results dict.
    """
    if not RED_FLAG_CONFIG.get('enabled', False):
        return {'triggered': {}, 'screening_answers': {}, 'referrals': {}}

    bonus = RED_FLAG_CONFIG['bonus']
    top_n = RED_FLAG_CONFIG['screening_top_n']

    # Budget: max total screening questions across all conditions
    _screening_budget = QUESTION_BUDGET.get('screening_max', 999)
    _screening_asked = 0

    # Get confirmed symptom root names and variant names for matching
    confirmed_info = nodes_symptom[nodes_symptom['uuid'].isin(confirmed_uuids)]
    confirmed_roots = set(confirmed_info['root_snomed_name'].dropna().unique())
    confirmed_names = set(confirmed_info['name'].dropna().unique())

    # Top-N condition IDs from result_df (pre-bonus ranking)
    top_n_cids = set(result_df.head(top_n)['condition_snomed_id'].tolist())

    triggered = {}       # cid -> list of triggered flag names
    screening_answers = {} # cid -> list of {'flag': ..., 'answer': bool}
    referrals = {}       # cid -> referral level string

    print(f"\n{'=' * 50}")
    print("  RED FLAG SCREENING")
    print(f"{'=' * 50}")

    for cid, rf_entry in RED_FLAG_MAP.items():
        if cid not in surviving_pool:
            continue

        cond_name = rf_entry['name']
        cond_flags = []

        # --- Phase 1: Check triggers from confirmed symptoms ---
        for trigger in rf_entry.get('triggers', []):
            flag_name = trigger['flag']
            match_roots = set(trigger.get('match_roots', []))
            match_variants = set(trigger.get('match_variants', []))

            # Check variant-level match first (more specific)
            if match_variants and match_variants & confirmed_names:
                cond_flags.append(flag_name)
            elif match_roots and match_roots & confirmed_roots:
                cond_flags.append(flag_name)

        # --- Phase 2: Screening questions (only if condition is in top-N) ---
        screening_qs = rf_entry.get('screening_questions', [])
        cond_screening = []
        if screening_qs and cid in top_n_cids:
            print(f"\n  [Screening: {cond_name}]")
            for sq in screening_qs:
                if _screening_asked >= _screening_budget:
                    print(f"     (screening budget reached: {_screening_budget} Qs)")
                    break
                flag_name = sq['flag']
                question = sq['question']
                answer = input(f"  >> {question} (yes/no): ").strip().lower()
                _screening_asked += 1
                answered_yes = answer in ('yes', 'y', 'yeah', 'ya')
                cond_screening.append({'flag': flag_name, 'answer': answered_yes})
                if answered_yes:
                    cond_flags.append(flag_name)
                    print(f"     -> RED FLAG: {flag_name}")
                else:
                    print(f"     -> No concern: {flag_name}")

        if cond_flags:
            triggered[cid] = cond_flags
        if cond_screening:
            screening_answers[cid] = cond_screening

        # --- Phase 3: Determine referral level (GERD only for now) ---
        if rf_entry.get('referral_rules') and cond_flags:
            rules = rf_entry['referral_rules']
            referral_level = 'routine'

            # Check emergency
            if 'emergency' in rules:
                e_rule = rules['emergency']
                e_any = set(e_rule.get('any_flags', []))
                if e_any & set(cond_flags):
                    referral_level = 'emergency'
                combo = e_rule.get('combo')
                if combo and combo.get('require_all'):
                    if all(f in cond_flags for f in combo['flags']):
                        referral_level = 'emergency'

            # Check urgent (only if not already emergency)
            if referral_level != 'emergency' and 'urgent_clinic' in rules:
                u_rule = rules['urgent_clinic']
                u_any = set(u_rule.get('any_flags', []))
                if u_any & set(cond_flags):
                    referral_level = 'urgent_clinic'

            referrals[cid] = referral_level

    # --- Phase 4: Apply bonus to result_df ---
    bonus_applied = {}
    for cid, flags in triggered.items():
        n_flags = len(flags)
        pc = result_df.loc[result_df['condition_snomed_id'] == cid, 'pc_weight']
        if len(pc) > 0:
            pc_val = pc.values[0]
            total_bonus = n_flags * bonus * pc_val
            idx = result_df.index[result_df['condition_snomed_id'] == cid]
            result_df.loc[idx, 'final_score'] += total_bonus
            bonus_applied[cid] = {'n_flags': n_flags, 'bonus_total': round(total_bonus, 4)}

    # Re-sort by final_score
    result_df.sort_values('final_score', ascending=False, inplace=True)
    result_df.reset_index(drop=True, inplace=True)

    # --- Summary ---
    print(f"\n  Red flag summary: ({_screening_asked} screening Q(s) asked, budget={_screening_budget})")
    if triggered:
        for cid, flags in triggered.items():
            cname = RED_FLAG_MAP[cid]['name']
            ba = bonus_applied.get(cid, {})
            print(f"    {cname}: {len(flags)} flag(s) triggered -> +{ba.get('bonus_total', 0):.3f} bonus")
            for f in flags:
                print(f"      - {f}")
            if cid in referrals:
                print(f"      Referral: {referrals[cid].upper()}")
    else:
        print("    No red flags triggered.")

    return {
        'triggered': triggered,
        'screening_answers': screening_answers,
        'referrals': referrals,
        'bonus_applied': bonus_applied,
    }


print("Function: process_red_flags() defined.")


Function: process_red_flags() defined.


In [56]:

# confirmed_symptoms = ['47258974-a65c-11eb-8d02-1e003a340630']  # e.g. "Fever"
# surv_cond = set(edges_present_in[edges_present_in['symptom_uuid'].isin(confirmed_symptoms)]['condition_snomed_id'].unique().tolist())
# age = 35
# gender = 'male'

# condition_points = {cid: 0.0 for cid in surv_cond}
# denied = []
# rank_conditions(confirmed_symptoms, denied, surv_cond, condition_points, age, gender)[0].head(20)


---

## Test


In [57]:
print("=" * 70)
print("  BODHI SYMPTOM CHECKER v7 — Dual-Track + Red Flags")
print("=" * 70)
print()

symptom_input = input("What symptom are you experiencing? ").strip()
gender_input_global  = input("What is your gender? (M/F): ").strip()
age_input_global     = int(input("What is your age? ").strip())

print(f"\nReceived: symptom='{symptom_input}', gender='{gender_input_global}', age={age_input_global}")

print(f"\n{'=' * 70}")
print("  DONE: SYMPTOM EXPANSION")
print(f"{'=' * 70}")

expansion_result = symptom_expansion(symptom_input)

if expansion_result is None:
    print("Could not process symptom.")
else:

    print(f"\n{'=' * 70}")
    print("  DONE: DUAL-TRACK QUESTIONING")
    print(f"{'=' * 70}")

    (confirmed_uuids, denied_uuids, surviving_pool,
     condition_points, protection_counters, question_log) = ask_questions(expansion_result)


    print(f"\n{'=' * 70}")
    print("  DONE: CONDITION RANKING")
    print(f"{'=' * 70}")

    result_df, detail_df = rank_conditions(
        confirmed_uuids, denied_uuids, surviving_pool,
        condition_points, age_input_global, gender_input_global
    )

    print(f"\n{'=' * 70}")
    print("  DONE: RED FLAG SCREENING")
    print(f"{'=' * 70}")

    red_flag_results = process_red_flags(
        confirmed_uuids, denied_uuids, surviving_pool,
        condition_points, result_df
    )


  BODHI SYMPTOM CHECKER v7 — Dual-Track + Red Flags


Received: symptom='abdominal pain', gender='m', age=22

  DONE: SYMPTOM EXPANSION
Symptom: Abdominal pain (root_id: 21522001.0)
Variants: 51 | Conditions: 79 | Discovered symptoms: 206

  DONE: DUAL-TRACK QUESTIONING
  DUAL-TRACK QUESTIONING FOR: Abdominal pain
  Starting pool: 79 conditions
  Variant follow-up: ON (max 3 per symptom)
  Prerequisites: mode=pre_screen
  Root confirmed: Abdominal pain

  Q1 [follow-up variant] (score:3.640 | P(YES):3.640 | YES_elim:0 NO_elim:7)
    What type of pain is it?
      1. colicky pain
      2. crampy pain
      3. sharp or stabbing pain
      4. dull aching pain
      5. burning pain
    Connected to: Acute cholecystitis (2.5), Acute gastroenteritis (2.6), Appendicitis (2.6), Pica (2.4), Worm Infestation (1.9)...
    -> Confirmed: crampy pain
    -> Confirmed: dull aching pain
    Pool: 79 conditions remaining

  Q2 [follow-up variant] (score:3.540 | P(YES):3.540 | YES_elim:0 NO_elim:4)
    

In [14]:
# surviving_pool

### Results

In [58]:
if expansion_result is not None and len(result_df) > 0:

    print("=== Question Log ===\n")
    display(question_log)


    print(f"\n\n=== Top 5 Ranked Conditions ===\n")
    for i, row in result_df.head(5).iterrows():
        # Check if this condition has red flag bonus
        rf_bonus = red_flag_results.get('bonus_applied', {}).get(row['condition_snomed_id'], {})
        bonus_str = ""
        if rf_bonus:
            bonus_str = f" + RF_bonus({rf_bonus['bonus_total']:+.3f})"

        print(f"  {i+1}. {row['condition_name']}")
        print(f"     Final: {row['final_score']:.2f} = "
              f"Y/N({row['yn_points']:+.1f}) + P(C|S)({row['pcs_score']:.1f}) + "
              f"Age({row['age_weight']:.1f}) + Gen({row['gender_weight']:.1f}){bonus_str}")
        print(f"     Matches: {row['num_symptom_matches']} symptoms | Triage: {row['triage_level']}")

        # Show red flags if any
        cid = row['condition_snomed_id']
        flags = red_flag_results.get('triggered', {}).get(cid, [])
        if flags:
            print(f"     RED FLAGS: {', '.join(flags)}")
        referral = red_flag_results.get('referrals', {}).get(cid)
        if referral:
            print(f"     REFERRAL: {referral.upper()}")
        print()


    print("\n=== Full Ranking ===")
    display(result_df)


    confirmed_symptoms = nodes_symptom[nodes_symptom['uuid'].isin(confirmed_uuids)][
        ['uuid', 'root_snomed_name', 'name', 'triage_level']
    ].reset_index(drop=True)
    print(f"\n=== Confirmed Symptoms ({len(confirmed_symptoms)}) ===")
    display(confirmed_symptoms)


    top5_names = result_df.head(5)['condition_name'].tolist()
    top5_detail = detail_df[detail_df['condition_name'].isin(top5_names)]
    print(f"\n=== Scoring Detail (Top 5) ===")
    display(top5_detail)


    total_conditions = len(expansion_result['condition_ids'])
    eliminated_count = total_conditions - len(surviving_pool)
    print(f"\n=== Elimination Summary ===")
    print(f"Started with: {total_conditions} conditions")
    print(f"Eliminated:   {eliminated_count}")
    print(f"Survived:     {len(surviving_pool)}")
    print(f"Questions asked: {len(question_log)}")


    # --- Red Flag Summary ---
    if red_flag_results.get('triggered'):
        print(f"\n=== Red Flag Summary ===")
        for cid, flags in red_flag_results['triggered'].items():
            cname = RED_FLAG_MAP[cid]['name']
            ba = red_flag_results.get('bonus_applied', {}).get(cid, {})
            print(f"  {cname}:")
            print(f"    Flags triggered: {len(flags)}")
            for f in flags:
                print(f"      - {f}")
            if ba:
                print(f"    Bonus applied: +{ba['bonus_total']:.3f}")
            referral = red_flag_results.get('referrals', {}).get(cid)
            if referral:
                print(f"    Referral level: {referral.upper()}")

    if red_flag_results.get('screening_answers'):
        print(f"\n=== Red Flag Screening Answers ===")
        for cid, answers in red_flag_results['screening_answers'].items():
            cname = RED_FLAG_MAP[cid]['name']
            print(f"  {cname}:")
            for a in answers:
                status = "YES" if a['answer'] else "NO"
                print(f"    [{status}] {a['flag']}")


    if protection_counters:
        print(f"\n=== Protection Counters ===")
        for cid, counter in sorted(protection_counters.items(), key=lambda x: x[1], reverse=True):
            status = "IN POOL" if cid in surviving_pool else "ELIMINATED"
            print(f"  {get_condition_name(cid):40s} counter={counter:.1f} ({status})")


=== Question Log ===



,order,type,symptom,relation_type,score,p_yes,yes_elim,no_elim,connected_conditions,pool_after,q_number,root_name,symptom_uuid,answer
0,1.0,follow-up variant,Abdominal pain,pain_type,3.640,3.640,0,7,"Acute cholecystitis, Acute gastroenteritis, Ap...",79,NaN,NaN,NaN,NaN
1,2.0,follow-up variant,Abdominal pain,location,3.540,3.540,0,4,"Irritable bowel syndrome, Vaginitis, High alti...",79,NaN,NaN,NaN,NaN
2,3.0,follow-up variant,Abdominal pain,onset,1.897,1.897,0,4,"Acute cholecystitis, Acute gastroenteritis, Ac...",75,NaN,NaN,NaN,NaN
3,4.0,prerequisite,Intake of drug,prerequisite,1.000,0.000,0,1,Fixed drug eruptions,74,NaN,NaN,NaN,NaN
4,5.0,prerequisite,Diabetes mellitus type 1,prerequisite,2.000,0.000,0,2,"Diabetic neuropathy, Diabetic Ketoacidosis",72,NaN,NaN,NaN,NaN
5,6.0,prerequisite,High altitude,prerequisite,1.000,0.000,0,1,High altitude sickness,71,NaN,NaN,NaN,NaN
6,7.0,discovered,Fever,-,36.835,36.835,0,15,"Vaginitis, Acute gastroenteritis, Crohn's Dise...",56,NaN,NaN,NaN,NaN
7,8.0,discovered,Fatigue,-,33.285,33.285,0,11,"Diabetes mellitus type 1, Hypoparathyroidism, ...",45,NaN,NaN,NaN,NaN
8,9.0,discovered,Headache,-,25.825,25.825,0,1,"Hypoparathyroidism, Adrenal insufficiency, Wil...",44,NaN,NaN,NaN,NaN
9,NaN,adaptive-extra,NaN,NaN,22.140,NaN,0,0,"Irritable bowel syndrome, Chlamydial genitouri...",44,10.0,Malaise,4781953e-a65c-11eb-8d02-1e003a340630,yes




=== Top 5 Ranked Conditions ===

  1. Gastroesophageal reflux disease
     Final: 1.98 = Y/N(+0.2) + P(C|S)(0.8) + Age(1.0) + Gen(1.0) + RF_bonus(+1.300)
     Matches: 2 symptoms | Triage: opd_managed
     RED FLAGS: Vomiting, Odynophagia (pain on swallowing)
     REFERRAL: URGENT_CLINIC

  2. Gastritis
     Final: 0.88 = Y/N(+0.2) + P(C|S)(1.0) + Age(0.9) + Gen(1.0)
     Matches: 2 symptoms | Triage: opd_managed

  3. Chronic Pancreatitis
     Final: 0.69 = Y/N(+0.3) + P(C|S)(1.2) + Age(0.9) + Gen(1.0)
     Matches: 4 symptoms | Triage: worrisome

  4. Irritable bowel syndrome
     Final: 0.69 = Y/N(+0.2) + P(C|S)(1.2) + Age(1.0) + Gen(1.0)
     Matches: 3 symptoms | Triage: opd_managed

  5. Kidney stone
     Final: 0.68 = Y/N(+0.5) + P(C|S)(0.6) + Age(1.0) + Gen(1.0)
     Matches: 2 symptoms | Triage: worrisome


=== Full Ranking ===


,condition_snomed_id,condition_name,triage_level,type_condition,num_symptom_matches,total_roots,match_coverage,yn_points,pcs_score,pc_weight,gender_weight,age_weight,coverage_bonus,final_score
0,235595009,Gastroesophageal reflux disease,opd_managed,chronic_with_acute_aggravation,2,10,0.200,0.20,0.85,0.65,1.00,1.00,0,1.9825
1,4556007,Gastritis,opd_managed,acute_that_may_turn_chronic,2,9,0.222,0.22,1.00,0.80,1.00,0.90,0,0.8800
2,235494005,Chronic Pancreatitis,worrisome,chronic,4,7,0.571,0.29,1.25,0.50,1.00,0.90,0,0.6911
3,10743008,Irritable bowel syndrome,opd_managed,chronic,3,11,0.273,0.18,1.20,0.50,1.00,1.00,0,0.6909
4,95570007,Kidney stone,worrisome,acute,2,6,0.333,0.50,0.55,0.65,1.00,1.00,0,0.6825
5,197321007,Fatty liver,opd_managed,acute_that_may_turn_chronic,3,8,0.375,0.25,1.20,0.80,1.00,0.50,0,0.5800
6,34000006,Crohn's Disease,worrisome,chronic,3,17,0.176,-0.06,1.20,0.50,0.90,1.00,0,0.5135
7,397825006,Gastric ulcer,worrisome,chronic_with_acute_aggravation,2,9,0.222,0.22,0.85,0.50,1.00,0.50,0,0.2681
8,396232000,Inguinal Hernia,worrisome,chronic,2,6,0.333,0.33,0.70,0.50,1.00,0.50,0,0.2583
9,417357006,Sickle cell anaemia,emergency,chronic,2,33,0.061,0.03,0.70,0.35,1.00,1.00,0,0.2556



=== Confirmed Symptoms (6) ===


,uuid,root_snomed_name,name,triage_level
0,47172c6c-a65c-11eb-8d02-1e003a340630,Abdominal pain,Abdominal pain,opd_managed
1,4718acf4-a65c-11eb-8d02-1e003a340630,Abdominal pain,Abdominal pain <pain> crampy pain,worrisome
2,471907ee-a65c-11eb-8d02-1e003a340630,Abdominal pain,Abdominal pain <pain> dull aching pain,worrisome
3,4719de80-a65c-11eb-8d02-1e003a340630,Abdominal pain,Abdominal pain <loc> diffuse,worrisome
4,4737ee0c-a65c-11eb-8d02-1e003a340630,Vomit,Vomit,worrisome
5,4781953e-a65c-11eb-8d02-1e003a340630,Malaise,Malaise,opd_managed



=== Scoring Detail (Top 5) ===


,symptom_variant,root_symptom,condition_name,likelihood_text,pcs_score
10,Vomit,Vomit,Chronic Pancreatitis,low,0.35
11,Abdominal pain <pain> crampy pain,Abdominal pain,Chronic Pancreatitis,low,0.35
12,Abdominal pain,Abdominal pain,Chronic Pancreatitis,low,0.35
13,Malaise,Malaise,Chronic Pancreatitis,rare,0.20
29,Abdominal pain,Abdominal pain,Gastritis,high,0.65
30,Vomit,Vomit,Gastritis,low,0.35
31,Abdominal pain,Abdominal pain,Gastroesophageal reflux disease,medium,0.50
32,Vomit,Vomit,Gastroesophageal reflux disease,low,0.35
48,Abdominal pain <loc> diffuse,Abdominal pain,Irritable bowel syndrome,medium,0.50
49,Malaise,Malaise,Irritable bowel syndrome,low,0.35



=== Elimination Summary ===
Started with: 79 conditions
Eliminated:   35
Survived:     44
Questions asked: 11

=== Red Flag Summary ===
  Gastroesophageal reflux disease:
    Flags triggered: 2
      - Vomiting
      - Odynophagia (pain on swallowing)
    Bonus applied: +1.300
    Referral level: URGENT_CLINIC

=== Red Flag Screening Answers ===
  Gastroesophageal reflux disease:
    [YES] Odynophagia (pain on swallowing)
    [NO] Unexplained weight loss

=== Protection Counters ===
  Puerperal Sepsis                         counter=0.2 (ELIMINATED)


In [49]:
nc = nodes_condition[['snomed_id','overall_likelihood']]
lkscoremap = {'very_high': 0.8,
 'high': 0.65,
 'medium': 0.5,
 'low': 0.35,
 'rare': 0.2,
 'zero': 0.0}

nc['overall_likelihood_score'] = nc['overall_likelihood'].apply(lkscoremap.get)

nc2 = edges_present_in.merge(nc,left_on = 'condition_snomed_id',
                        right_on = 'snomed_id',
                         how = 'left')


nc2['likelihood_symptom_given_condition_score'] = nc2['likelihood_symptom_given_condition'].map(lkscoremap.get)

nc2['symptom_score'] = nc2['likelihood_symptom_given_condition_score'] * nc2['overall_likelihood_score']

nc3 = (nc2.groupby(['symptom_uuid'],as_index=False).agg({'symptom_score': 'sum',
                                                   'condition_snomed_id':'nunique'}).sort_values('symptom_score',ascending=False)
 .merge(nodes_symptom[['uuid','name']],left_on = 'symptom_uuid',
        right_on = 'uuid',how = 'left'))


# nc3['symptom_score_normalized'] = nc3['symptom_score'] / nc3['condition_snomed_id']

nc3.sort_values('symptom_score',ascending=False).head(10)

,symptom_uuid,symptom_score,condition_snomed_id,uuid,name
0,47258974-a65c-11eb-8d02-1e003a340630,36.8350,145,47258974-a65c-11eb-8d02-1e003a340630,Fever
1,473914a8-a65c-11eb-8d02-1e003a340630,33.2850,126,473914a8-a65c-11eb-8d02-1e003a340630,Fatigue
2,4720607a-a65c-11eb-8d02-1e003a340630,25.8250,109,4720607a-a65c-11eb-8d02-1e003a340630,Headache
3,4781953e-a65c-11eb-8d02-1e003a340630,22.1400,81,4781953e-a65c-11eb-8d02-1e003a340630,Malaise
4,4737ee0c-a65c-11eb-8d02-1e003a340630,21.7550,89,4737ee0c-a65c-11eb-8d02-1e003a340630,Vomit
5,47172c6c-a65c-11eb-8d02-1e003a340630,19.3375,79,47172c6c-a65c-11eb-8d02-1e003a340630,Abdominal pain
6,47483014-a65c-11eb-8d02-1e003a340630,15.8725,67,47483014-a65c-11eb-8d02-1e003a340630,Nausea
7,471e8e62-a65c-11eb-8d02-1e003a340630,13.9500,48,471e8e62-a65c-11eb-8d02-1e003a340630,Cough
8,4728ddae-a65c-11eb-8d02-1e003a340630,12.8925,51,4728ddae-a65c-11eb-8d02-1e003a340630,Joint pain
9,473ca9f6-a65c-11eb-8d02-1e003a340630,12.7650,57,473ca9f6-a65c-11eb-8d02-1e003a340630,Breathlessness
